# 🐖 01 | Auditoría de imágenes y anotaciones

**RECURSO REQUERIDO: CPU (sin acelerador)**. Este notebook realiza exclusivamente su propia etapa. **No cambies CPU/GPU dentro del notebook.**

Proyecto: Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖. Datos: FaroPigSeg, licencia reportada CC BY-NC 4.0; citar fuente https://github.com/yudong888/FaroPig-Datasets.

**Ruta general:** `MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖/`. Ejecutar en orden 00 → 01 → 02 → 03 → 04. Todos los resultados se guardan en Drive.

## Objetivo
Inspeccionar integridad de anotaciones YOLO-seg, imágenes, grupos temporales y posibles solapamientos de fechas en las particiones originales. Genera **Excel**, gráfico y JSON. **No entrena ni cambia las particiones**.

In [ ]:
%pip -q install pandas openpyxl pillow matplotlib numpy tqdm

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, os

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError(
        'No se encontró la carpeta principal. Verifica la cuenta de Google y el nombre de la carpeta: '
        'Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖'
    )
DATA_ROOT = ROOT / '01_Datasets' / 'FaroPigSeg' / 'extraido' / 'FaroPigSeg'
DATASET_DIR = ROOT / '01_Datasets' / 'FaroPigSeg'
RES = ROOT / '03_Resultados'
EXPO = ROOT / '05_Exportaciones_ZIP'
for sub in ('01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento'):
    (RES / sub).mkdir(parents=True, exist_ok=True)
EXPO.mkdir(parents=True,exist_ok=True)
print('Proyecto:',ROOT)

In [ ]:
import re, numpy as np, pandas as pd
from PIL import Image
from collections import defaultdict
from tqdm.auto import tqdm

for s in ('train','val','test'):
    for k in ('images','labels'):
        assert (DATA_ROOT/s/k).is_dir(), f'Falta {DATA_ROOT/s/k}: ejecuta antes el notebook 00.'

imagenes=[]; incidencias=[]; instancias=[]
exts={'.jpeg','.jpg','.png','.webp','.bmp'}
pat=re.compile(r'^(\d{4}-\d{2}-\d{2})')
for split in ('train','val','test'):
    fotos=sorted(x for x in (DATA_ROOT/split/'images').iterdir() if x.suffix.lower() in exts)
    for img in tqdm(fotos,desc='Auditando '+split):
        fecha=pat.match(img.name)
        etiqueta=DATA_ROOT/split/'labels'/(img.stem+'.txt')
        try:
            with Image.open(img) as im:
                width,height=im.size
                im.verify()
        except Exception as e:
            incidencias.append({'imagen':img.name,'particion':split,'error':'imagen_ilegible:'+str(e)})
            continue
        validas=0
        if not fecha:incidencias.append({'imagen':img.name,'particion':split,'error':'fecha_no_identificada'})
        if not etiqueta.exists():
            incidencias.append({'imagen':img.name,'particion':split,'error':'falta_etiqueta'})
        else:
            for ln,linea in enumerate(etiqueta.read_text(encoding='utf-8-sig').splitlines(),1):
                if not linea.strip():continue
                try:
                    tok=linea.split();clase=int(tok[0]);coords=np.array([float(x) for x in tok[1:]],dtype=float)
                    if clase!=0 or len(coords)<6 or len(coords)%2 or not np.all(np.isfinite(coords)) or not np.all((coords>=0)&(coords<=1)):
                        raise ValueError('Clase o coordenadas no válidas')
                    xy=coords.reshape(-1,2)*[width,height]
                    area=0.5*abs(np.dot(xy[:,0],np.roll(xy[:,1],-1))-np.dot(xy[:,1],np.roll(xy[:,0],-1)))
                    if area<=0: raise ValueError('Área poligonal nula')
                    instancias.append({'particion':split,'imagen':img.name,'fecha':fecha.group(1) if fecha else None,
                                       'area_px2':float(area),'vertices':len(xy)})
                    validas+=1
                except Exception as e:
                    incidencias.append({'imagen':img.name,'particion':split,'linea':ln,'error':str(e)})
        imagenes.append({'particion':split,'imagen':img.name,'fecha':fecha.group(1) if fecha else None,
                         'ancho_px':width,'alto_px':height,'instancias_validas':validas,
                         'ruta_relativa':str(img.relative_to(DATA_ROOT))})

DF=pd.DataFrame(imagenes);INS=pd.DataFrame(instancias);ERR=pd.DataFrame(incidencias)
assert not DF.empty,'No se encontraron imágenes.'
RESUMEN=(DF.groupby('particion').agg(imagenes=('imagen','size'),instancias=('instancias_validas','sum'),
                                 fechas=('fecha','nunique')).reset_index())
fechas={s:set(DF.loc[DF.particion.eq(s),'fecha'].dropna()) for s in ('train','val','test')}
SOLAP=pd.DataFrame([{'particion_A':a,'particion_B':b,'fechas_compartidas':len(fechas[a]&fechas[b]),
                     'fechas':'; '.join(sorted(fechas[a]&fechas[b]))}
                    for a,b in [('train','val'),('train','test'),('val','test')]])
print(RESUMEN.to_string(index=False)); print('\nSOLAPAMIENTO DE FECHAS:');print(SOLAP[['particion_A','particion_B','fechas_compartidas']].to_string(index=False))
print('Incidencias:',len(ERR))
if len(ERR):display(ERR.head(12))

In [ ]:
import matplotlib.pyplot as plt
XLS=RES/'01_Exploracion'/'CERDOS_01_Auditoria.xlsx'
with pd.ExcelWriter(XLS,engine='openpyxl') as writer:
    RESUMEN.to_excel(writer,sheet_name='Resumen',index=False)
    DF.to_excel(writer,sheet_name='Imagenes',index=False)
    INS.to_excel(writer,sheet_name='Poligonos_GT',index=False)
    SOLAP.to_excel(writer,sheet_name='Solapamiento_fechas',index=False)
    (ERR if len(ERR) else pd.DataFrame({'estado':['Sin incidencias']})).to_excel(writer,sheet_name='Incidencias',index=False)
fig,ax=plt.subplots(figsize=(7,4)); ax.bar(RESUMEN.particion,RESUMEN.imagenes)
ax.set(title='FaroPigSeg: imágenes en particiones originales',ylabel='Imágenes')
for i,v in enumerate(RESUMEN.imagenes):ax.text(i,v,str(v),ha='center',va='bottom')
fig.tight_layout();PNG=RES/'04_Figuras'/'CERDOS_01_Distribucion_Original.png';fig.savefig(PNG,dpi=180);plt.show()
metadata={'etapa':'01_auditoria_CPU','fecha_utc':datetime.now(timezone.utc).isoformat(),
          'numero_imagenes':int(len(DF)),'instancias_validas':int(len(INS)),
          'incidencias':int(len(ERR)), 'solapamiento_fecha':SOLAP.to_dict('records'),
          'advertencia':'Las fechas de archivo son aproximaciones de sesiones; no prueba independencia por granja.'}
report=RES/'05_Reportes'/'CERDOS_01_Auditoria.json'
report.write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf8')
print('✅ Excel:',XLS,'\n✅ Reporte:',report,'\n✅ Figura:',PNG)

**Siguiente:** ejecutar `02_CERDOS_Preparacion_Segmentacion_CPU.ipynb` con **CPU** para construir una partición sin solapamientos de fechas.